# 05. So sánh model và phân tích kết quả

Notebook này đọc các kết quả đã tạo ở notebook 04 để chọn model tốt nhất, so sánh hai scenario, đánh giá trên test, phân tích lỗi và tạo các hình dùng trong báo cáo.

Nguyên tắc: test chỉ được dùng cho đánh giá cuối cùng, không dùng để chọn model hoặc chỉnh tham số.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
required_files = ['validation_results.csv', 'test_predictions.csv', 'metrics.json', 'best_regression_model.pkl']
missing_files = [name for name in required_files if not (ARTIFACTS_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError('Thiếu artifact {}. Hãy chạy notebook 04 trước.'.format(missing_files))

validation_results = pd.read_csv(ARTIFACTS_DIR / 'validation_results.csv')
test_predictions = pd.read_csv(ARTIFACTS_DIR / 'test_predictions.csv')
metrics_payload = json.loads((ARTIFACTS_DIR / 'metrics.json').read_text(encoding='utf-8'))
print('Đã đọc {} kết quả validation.'.format(len(validation_results)))

In [ ]:
print('Bảng xếp hạng theo validation RMSE:')
display(validation_results.sort_values(['rmse', 'mae']).reset_index(drop=True))

scenario_comparison = validation_results.groupby('scenario')[['mae', 'rmse', 'r2']].mean().sort_values('rmse')
print('Trung bình kết quả theo scenario:')
display(scenario_comparison)

best_validation = validation_results.sort_values(['rmse', 'mae']).iloc[0]
print('Model tốt nhất theo validation: {} - {}'.format(best_validation['model'], best_validation['scenario']))
print('Metrics cuối cùng trên test:')
display(pd.Series(metrics_payload['test_metrics'], name='test_value').to_frame())

In [ ]:
# Biểu đồ so sánh các model trên validation.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, title in zip(axes, ['mae', 'rmse', 'r2'], ['MAE thấp hơn là tốt', 'RMSE thấp hơn là tốt', 'R2 cao hơn là tốt']):
    plot_df = validation_results.copy()
    plot_df['label'] = plot_df['scenario'] + ' / ' + plot_df['model']
    sns.barplot(data=plot_df, y='label', x=metric, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(metric.upper())
    ax.set_ylabel('')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'model_comparison_validation.png', dpi=150)
plt.show()

In [ ]:
# Phân tích những dự đoán sai nhiều nhất trên test.
top_errors = test_predictions.sort_values('absolute_error', ascending=False).head(10)
display(top_errors)

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=test_predictions, x='G3', y='predicted_G3', hue='subject', alpha=0.7, ax=ax)
line_min = min(test_predictions['G3'].min(), test_predictions['predicted_G3'].min())
line_max = max(test_predictions['G3'].max(), test_predictions['predicted_G3'].max())
ax.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='red')
ax.set_title('Giá trị thật và giá trị dự đoán trên test')
ax.set_xlabel('G3 thật')
ax.set_ylabel('G3 dự đoán')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'test_actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Xem các feature quan trọng của model tốt nhất.
best_pipeline = joblib.load(ARTIFACTS_DIR / 'best_regression_model.pkl')
preprocessor = best_pipeline.named_steps['preprocessor']
model = best_pipeline.named_steps['model']
feature_names = preprocessor.get_feature_names_out()

if hasattr(model, 'feature_importances_'):
    importance_values = model.feature_importances_
elif hasattr(model, 'coef_'):
    importance_values = np.abs(np.asarray(model.coef_).ravel())
else:
    importance_values = None

if importance_values is not None:
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance_values}).sort_values('importance', ascending=False)
    display(importance_df.head(15))
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=importance_df.head(15), y='feature', x='importance', ax=ax)
    ax.set_title('15 feature quan trọng nhất')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'best_model_feature_importance.png', dpi=150)
    plt.show()
else:
    importance_df = pd.DataFrame()
    print('Model không cung cấp feature_importances_ hoặc coef_.')

In [ ]:
# Lưu bản tóm tắt cuối cùng để dùng cho báo cáo.
final_summary = {
    'selected_model': str(metrics_payload['best_model']),
    'selected_scenario': str(metrics_payload['best_scenario']),
    'validation_rmse': float(best_validation['rmse']),
    'validation_mae': float(best_validation['mae']),
    'validation_r2': float(best_validation['r2']),
    'test_metrics': metrics_payload['test_metrics'],
    'largest_absolute_errors': top_errors[['subject', 'G3', 'predicted_G3', 'absolute_error']].to_dict(orient='records')
}
(ARTIFACTS_DIR / 'final_summary.json').write_text(json.dumps(final_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu final_summary.json và các biểu đồ vào artifacts/.')

## Nội dung cần đưa vào báo cáo

Nhóm cần ghi: model nào tốt nhất, scenario nào tốt hơn, các metric validation/test, biểu đồ dự đoán so với thực tế, feature importance và một vài trường hợp dự đoán sai. Không chỉ ghi mỗi con số; cần giải thích ý nghĩa và hạn chế của kết quả.